In [1]:
# !pip install wandb

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from src.preprocess import BasicDegenderizer, AdvancedDegenderizer
from src.models import DistilBERTClassifier, RoBERTaClassifier

In [4]:
import wandb

In [5]:
wandb.login(key="5489aee4351c1c3af108d0f20e5191f366756c2c")

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hice1/mwesley32/.netrc
wandb: Currently logged in as: mtwesley to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:
wandb.init(
    project="NLP-Letters-V2",
    group="nlp-letters-v2-distilbert-gendered"
)

In [7]:
DATA_PATH = "data/sentence_sets_trimmed.csv"
LABEL_COLUMN = "applicant_gender"
TEXT_COLUMN = "full_text"
DEGENDERIZERS = ["src/degender/20240301-all.txt"]

In [8]:
# Load dataset
df = pd.read_csv(DATA_PATH, encoding="ISO-8859-1")
print("Dataset shape:", df.shape)

Dataset shape: (3285, 19)


In [9]:
advanced_pipeline = Pipeline([("advanced", AdvancedDegenderizer(paths=DEGENDERIZERS))])
df["degendered"] = advanced_pipeline.fit_transform(df[TEXT_COLUMN].tolist())
df["gendered"] = df[TEXT_COLUMN]

In [10]:
# Convert categorical labels to factors / integers
df[LABEL_COLUMN], class_mapping = pd.factorize(df[LABEL_COLUMN])
print("Class mapping:", dict(enumerate(class_mapping)))

Class mapping: {0: 'male', 1: 'female'}


In [15]:
# Train test splits
X_train, X_test, y_train, y_test = train_test_split(
    df["degendered"],
    df[LABEL_COLUMN],
    test_size=0.2,
    stratify=df[LABEL_COLUMN],
)

print("Train size:", len(X_train), "Test size:", len(X_test))

Train size: 2628 Test size: 657


In [16]:
# DistilBERT classifier model
model = DistilBERTClassifier(
    model_name="distilbert-base-uncased",
    num_labels=len(class_mapping),
)

model.train(
    X_train.tolist(),
    y_train.tolist(),
    epochs=3,
    batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    output_dir="../scratch/nlp-letters-v2-distilbert-gendered",
    report_to=["wandb"]
)

Map:   0%|          | 0/2628 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Mcc,Balanced Accuracy,Cohen Kappa,Jaccard,Hamming Loss,Auc Roc,Auc Pr,0 Precision,0 Recall,0 F1,1 Precision,1 Recall,1 F1,Cm 00,Cm 01,Cm 10,Cm 11,Runtime,Samples Per Second,Steps Per Second
1,0.207900,0.204461,0.942966,0.936936,0.922794,0.929483,0.859614,0.922794,0.859006,0.869808,0.057034,0.922794,0.844281,0.950262,0.970588,0.960317,0.923611,0.875000,0.898649,363,11,19,133,3.041500,172.941000,10.850000
2,0.196400,0.123907,0.950570,0.948393,0.930094,0.938632,0.878297,0.930094,0.877319,0.885587,0.049430,0.930094,0.866133,0.953125,0.978610,0.965699,0.943662,0.881579,0.911565,366,8,18,134,3.046200,172.673000,10.833000
3,0.110300,0.136883,0.952471,0.953869,0.929479,0.940619,0.883011,0.929479,0.881328,0.889079,0.047529,0.929479,0.873352,0.950904,0.983957,0.967148,0.956835,0.875000,0.914089,368,6,19,133,3.043200,172.844000,10.844000


{'eval_loss': 0.13688288629055023,
 'eval_accuracy': 0.9524714828897338,
 'eval_precision': 0.9538694625694792,
 'eval_recall': 0.9294786096256684,
 'eval_f1': 0.9406189179547619,
 'eval_mcc': 0.8830112699980434,
 'eval_balanced_accuracy': 0.9294786096256684,
 'eval_cohen_kappa': 0.8813283999639021,
 'eval_jaccard': 0.8890794601732857,
 'eval_hamming_loss': 0.04752851711026616,
 'eval_auc_roc': 0.9294786096256684,
 'eval_auc_pr': 0.8733518888311405,
 'eval_0_precision': 0.9509043927648578,
 'eval_0_recall': 0.983957219251337,
 'eval_0_f1': 0.9671484888304862,
 'eval_1_precision': 0.9568345323741008,
 'eval_1_recall': 0.875,
 'eval_1_f1': 0.9140893470790378,
 'eval_cm_00': 368,
 'eval_cm_01': 6,
 'eval_cm_10': 19,
 'eval_cm_11': 133,
 'eval_runtime': 3.0418,
 'eval_samples_per_second': 172.923,
 'eval_steps_per_second': 10.849,
 'epoch': 3.0}

In [17]:
# Inference on de-gendered
model.test(df["degendered"], df[LABEL_COLUMN])

Map:   0%|          | 0/3285 [00:00<?, ? examples/s]

{'accuracy': 0.9616438356164384,
 'precision': 0.9635774139868458,
 'recall': 0.9416895896780955,
 'f1': 0.9518257584786478,
 'mcc': 0.9050023597049512,
 'balanced_accuracy': 0.9416895896780955,
 'cohen_kappa': 0.9037064333901449,
 'jaccard': 0.9089014917071798,
 'hamming_loss': 0.038356164383561646,
 'auc_roc': 0.9416895896780955,
 'auc_pr': 0.8961844011212862,
 '0_precision': 0.9594873914840844,
 '0_recall': 0.9880800340570456,
 '0_f1': 0.9735738255033557,
 '1_precision': 0.9676674364896074,
 '1_recall': 0.8952991452991453,
 '1_f1': 0.93007769145394,
 'cm_00': 2321,
 'cm_01': 28,
 'cm_10': 98,
 'cm_11': 838}

In [18]:
# Evaluation
model.test(X_test, y_test)

Map:   0%|          | 0/657 [00:00<?, ? examples/s]

{'accuracy': 0.9421613394216134,
 'precision': 0.9384584149429132,
 'recall': 0.9177153259756514,
 'f1': 0.9273060796645702,
 'mcc': 0.855922425748143,
 'balanced_accuracy': 0.9177153259756514,
 'cohen_kappa': 0.8546950843334226,
 'jaccard': 0.8662161614524233,
 'hamming_loss': 0.0578386605783866,
 'auc_roc': 0.9177153259756513,
 'auc_pr': 0.8408164404074844,
 '0_precision': 0.9462809917355371,
 '0_recall': 0.9744680851063829,
 '0_f1': 0.960167714884696,
 '1_precision': 0.930635838150289,
 '1_recall': 0.8609625668449198,
 '1_f1': 0.8944444444444445,
 'cm_00': 458,
 'cm_01': 12,
 'cm_10': 26,
 'cm_11': 161}